# CardioIA — Fase 2 · Parte 1
## Extração de sintomas e sugestão de diagnóstico

Este notebook lê relatos escritos em linguagem natural, encontra neles as expressões
de um **mapa de conhecimento** (uma ontologia sintoma → doença) e sugere uma hipótese
diagnóstica, mostrando a evidência que sustenta cada sugestão.

É deliberadamente um sistema **baseado em regras**, não em aprendizado: com 10 relatos
não há dado suficiente para treinar nada, e a regra tem uma vantagem que o modelo
estatístico da Parte 2 não terá — ela consegue dizer *por que* chegou àquela conclusão.

**Entradas**
| Arquivo | Conteúdo |
|---|---|
| `data/relatos_pacientes.txt` | 10 relatos de sintomas (fictícios, escritos por nós) |
| `data/mapa_conhecimento.csv` | 79 regras `sintoma_1 \| sintoma_2 \| doenca_associada`, 8 doenças |

A lógica vive em `src/extracao.py` — o notebook importa e demonstra, para que o mesmo
código rode também pela linha de comando (`python src/extracao.py`).

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
from src.extracao import (
    carregar_mapa,
    carregar_relatos,
    encontrar_expressoes,
    normalizar,
    pontuar,
    relatorio,
    sugerir_diagnostico,
    testar_desafio,
)

pd.set_option("display.max_colwidth", 90)

---
## 1. Os relatos

Cada linha do `.txt` é um relato completo: **o que o paciente sente, quando começou e
como aquilo afeta a rotina** — os três elementos pedidos no enunciado. As frases são
inventadas por nós, mas o vocabulário e os quadros clínicos seguem os sinais catalogados
no [UCI Heart Disease Data Set](https://archive.ics.uci.edu/dataset/45/heart+disease) —
tipo de dor torácica, angina por esforço, achado do ECG em repouso.

In [2]:
relatos = carregar_relatos()
print(f"{len(relatos)} relatos carregados\n")
for i, r in enumerate(relatos, 1):
    print(f"{i:>2}. {r}\n")

10 relatos carregados

 1. Há dois dias estou com uma dor no peito que aperta e piora quando faço esforço físico; ontem tive que parar no meio da subida para casa e a dor só passou depois de uns dez minutos sentado.

 2. Sinto um cansaço constante há uma semana, mesmo depois de descansar, e minhas pernas estão inchadas no fim do dia a ponto de o sapato marcar; para dormir preciso de dois travesseiros, senão me falta o ar deitado.

 3. Desde ontem à noite sinto o coração disparado e batendo descompassado, como se falhasse uma batida, junto com uma tontura que me obriga a sentar; já aconteceu três vezes hoje enquanto eu estava parado.

 4. Tenho falta de ar para subir um lance de escada que eu subia sem problema até o mês passado, e às vezes vem junto um aperto no peito que melhora quando eu paro de andar.

 5. Acordei hoje às cinco da manhã com uma dor forte no peito que irradia para o braço esquerdo e para a mandíbula, com suor frio e enjoo; não passou com o remédio de estômago e já fa

---
## 2. O mapa de conhecimento

Cada linha é uma **regra**: duas expressões que, juntas, apontam para uma doença.
Manter três colunas — `sintoma_1 | sintoma_2 | doenca_associada` — é o formato pedido
no enunciado, e ele carrega mais informação do que uma lista simples de palavras-chave:
o par codifica *combinação* de sinais, que é como o raciocínio clínico funciona.
"Dor no peito" sozinha não decide nada; "dor no peito + piora ao esforço" aponta para
angina, e "dor no peito + suor frio" para infarto.

> **Divergência assumida em relação ao enunciado.** O exemplo do enunciado mapeia
> `"falta de ar", "dificuldade para respirar" → Angina`. Clinicamente, dispneia isolada
> é sinal de **insuficiência cardíaca**; angina é definida pela dor/aperto torácico
> desencadeado por esforço. Mantivemos a classificação clínica correta e registramos a
> divergência aqui em vez de reproduzir o exemplo.

In [3]:
mapa = carregar_mapa()
print(f"{len(mapa)} regras · {mapa.doenca_associada.nunique()} doenças\n")
display(mapa.doenca_associada.value_counts().to_frame("regras"))
display(mapa[["sintoma_1", "sintoma_2", "doenca_associada"]].head(8))

79 regras · 8 doenças



,regras
doenca_associada,
AVC,13
Infarto agudo do miocárdio,12
Insuficiência cardíaca,12
Angina,10
Estenose aórtica,10
Arritmia cardíaca,9
Hipertensão arterial,7
Pericardite,6


,sintoma_1,sintoma_2,doenca_associada
0,dor no peito,aperto no peito,Infarto agudo do miocárdio
1,dor forte no peito,suor frio,Infarto agudo do miocárdio
2,dor no peito,irradia para o braço esquerdo,Infarto agudo do miocárdio
3,dor no peito,dor no braço esquerdo,Infarto agudo do miocárdio
4,dor no peito,mandíbula,Infarto agudo do miocárdio
5,aperto no peito,suor frio,Infarto agudo do miocárdio
6,dor no peito,enjoo,Infarto agudo do miocárdio
7,dor no peito,náusea,Infarto agudo do miocárdio


---
## 3. Normalização: o passo que faz o casamento funcionar

O mapa guarda "não passou"; o paciente pode escrever "NÃO PASSOU", "nao passou" ou
"Não passou." no fim da frase. Antes de comparar qualquer coisa, texto e mapa passam
pela mesma função: minúsculas, acentos removidos, pontuação virando espaço. Sem isso, metade das regras nunca dispararia.

In [4]:
exemplo = "Acordei com uma DOR no peito — que irradia p/ o braço esquerdo!"
print("original: ", exemplo)
print("normalizado:", normalizar(exemplo))

original:  Acordei com uma DOR no peito — que irradia p/ o braço esquerdo!
normalizado: acordei com uma dor no peito que irradia p o braco esquerdo


---
## 4. Extração dos sintomas (com tratamento de negação)

`encontrar_expressoes` procura no relato todas as expressões do mapa. O detalhe que
evita o erro mais grosseiro: se a expressão vier logo depois de uma palavra de negação
(*não, nem, sem, nunca, nenhum*), ela é descartada. "Não sinto dor no peito" não pode contar
como dor no peito — e é exatamente esse o erro que o classificador estatístico da
Parte 2 vai cometer.

In [5]:
expressoes = set(mapa.expr_1) | set(mapa.expr_2)

com    = "sinto dor no peito quando subo escada"
sem    = "nao sinto dor no peito, so vim renovar a receita"
print("com sintoma :", sorted(encontrar_expressoes(com, expressoes)))
print("com negação :", sorted(encontrar_expressoes(sem, expressoes)))

com sintoma : ['dor no peito']
com negação : []


---
## 5. Como a hipótese é escolhida

Para cada doença:

$$\text{pontuação} = \underbrace{|\text{expressões distintas encontradas}|}_{\text{amplitude da evidência}} + 2 \times \underbrace{|\text{regras com as duas expressões}|}_{\text{evidência combinada}}$$

Contar **expressões distintas**, e não linhas do mapa, corrige um viés que aparece
quando se soma linha a linha: "dor no peito" aparece em 7 regras de infarto, então uma
queixa genérica venceria por repetição, sem nenhuma evidência específica. O bônus de 2
pontos premia a regra que disparou inteira — que é a informação que o mapa realmente
codifica.

**Evidência mínima.** Se nenhuma regra disparou inteira, o sistema **não sugere
diagnóstico**. Uma expressão solta como "dor no peito" está em três doenças com o mesmo
peso, e escolher entre elas seria decidir pelo desempate (a ordem alfabética), não pela
evidência. Nesse caso a saída é "sem hipótese", com as candidatas listadas.

O sistema devolve também as **alternativas** e avisa quando há empate. Um sistema de
triagem que devolve uma única resposta sem margem esconde do médico exatamente a
informação de que ele precisa.

In [6]:
caso = relatos[0]
print(caso, "\n")
display(pd.DataFrame([
    {"doença": h.doenca, "pontuação": h.pontuacao,
     "regras completas": h.regras_completas, "sintomas encontrados": ", ".join(h.sintomas)}
    for h in pontuar(caso, mapa)
]))

Há dois dias estou com uma dor no peito que aperta e piora quando faço esforço físico; ontem tive que parar no meio da subida para casa e a dor só passou depois de uns dez minutos sentado. 



,doença,pontuação,regras completas,sintomas encontrados
0,Angina,5,1,"dor no peito, esforco fisico, piora quando faco esforco"
1,Estenose aórtica,1,0,esforco fisico
2,Infarto agudo do miocárdio,1,0,dor no peito
3,Pericardite,1,0,dor no peito


In [7]:
# Evidência mínima: sintoma isolado não vira diagnóstico.
print(relatorio("estou sem ar e com dor no peito", mapa))

  Sintomas extraídos: dor no peito
  -> Sem hipótese: evidência insuficiente (nenhuma regra do mapa disparou inteira).
     Candidatas: Angina (1); Infarto agudo do miocárdio (1); Pericardite (1)


---
## 6. Os 10 relatos processados

In [8]:
for i, r in enumerate(relatos, 1):
    print(f"[Relato {i}] {r}")
    print(relatorio(r, mapa))
    print()

[Relato 1] Há dois dias estou com uma dor no peito que aperta e piora quando faço esforço físico; ontem tive que parar no meio da subida para casa e a dor só passou depois de uns dez minutos sentado.
  Sintomas extraídos: dor no peito, esforco fisico, piora quando faco esforco
  -> Hipótese: Angina (pontuação 5, 1 regra(s) completa(s))
     Alternativas: Estenose aórtica (1); Infarto agudo do miocárdio (1)

[Relato 2] Sinto um cansaço constante há uma semana, mesmo depois de descansar, e minhas pernas estão inchadas no fim do dia a ponto de o sapato marcar; para dormir preciso de dois travesseiros, senão me falta o ar deitado.
  Sintomas extraídos: cansaco, cansaco constante, dois travesseiros, mesmo depois de descansar, sapato marcar
  -> Hipótese: Insuficiência cardíaca (pontuação 7, 1 regra(s) completa(s))
     Alternativas: Estenose aórtica (1)

[Relato 3] Desde ontem à noite sinto o coração disparado e batendo descompassado, como se falhasse uma batida, junto com uma tontura que m

---
## 7. Avaliação: o extrator acerta o que deveria?

"Funciona" precisa ser medido, não afirmado. Anotamos o diagnóstico que cada relato foi
escrito para representar (o gabarito) e comparamos com a saída do sistema.

**O que este número é e o que não é:** os relatos e o mapa foram escritos pela mesma
equipe, então 100% aqui significa "as regras cobrem os casos que nós mesmos criamos" —
consistência interna, não desempenho clínico. Várias regras do mapa foram escritas
lendo esses relatos (`sapato marcar`, `dois travesseiros`, `ganhei três quilos`), então
o 10/10 era esperado. A seção 8 mede o que acontece com relatos que o mapa nunca viu.

In [9]:
gabarito = [
    "Angina", "Insuficiência cardíaca", "Arritmia cardíaca", "Angina",
    "Infarto agudo do miocárdio", "Hipertensão arterial", "AVC",
    "Pericardite", "Estenose aórtica", "Insuficiência cardíaca",
]

linhas = []
for r, esperado in zip(relatos, gabarito):
    # mesma regra de decisão do sistema (e do desafio da seção 8)
    hipotese = sugerir_diagnostico(r, mapa)
    previsto = hipotese.doenca if hipotese else "(sem hipótese)"
    outras = [h.pontuacao for h in pontuar(r, mapa) if h.doenca != hipotese.doenca] if hipotese else []
    margem = hipotese.pontuacao - max(outras, default=0) if hipotese else 0
    linhas.append({"relato": r[:55] + "…", "esperado": esperado,
                   "sugerido": previsto, "margem p/ a 2ª": margem,
                   "ok": "✔" if normalizar(previsto) == normalizar(esperado) else "✘"})

resultado = pd.DataFrame(linhas)
display(resultado)
acertos = (resultado.ok == "✔").sum()
print(f"Acertos: {acertos}/{len(resultado)} ({acertos/len(resultado):.0%})")
print(f"Menor margem para a segunda hipótese: {resultado['margem p/ a 2ª'].min():g} ponto(s)")

,relato,esperado,sugerido,margem p/ a 2ª,ok
0,Há dois dias estou com uma dor no peito que aperta e pi…,Angina,Angina,4,✔
1,"Sinto um cansaço constante há uma semana, mesmo depois …",Insuficiência cardíaca,Insuficiência cardíaca,6,✔
2,Desde ontem à noite sinto o coração disparado e batendo…,Arritmia cardíaca,Arritmia cardíaca,7,✔
3,Tenho falta de ar para subir um lance de escada que eu …,Angina,Angina,4,✔
4,Acordei hoje às cinco da manhã com uma dor forte no pei…,Infarto agudo do miocárdio,Infarto agudo do miocárdio,10,✔
5,Há três meses sinto dor de cabeça latejante na nuca qua…,Hipertensão arterial,Hipertensão arterial,8,✔
6,Ontem de manhã meu braço direito ficou dormente e fraco…,AVC,AVC,8,✔
7,Sinto uma dor no peito que piora quando eu respiro fund…,Pericardite,Pericardite,9,✔
8,Há um mês tenho tido tontura e escurecimento da vista q…,Estenose aórtica,Estenose aórtica,10,✔
9,Tenho tosse seca à noite há duas semanas e acordo sufoc…,Insuficiência cardíaca,Insuficiência cardíaca,6,✔


Acertos: 10/10 (100%)
Menor margem para a segunda hipótese: 4 ponto(s)


---
## 8. Relatos novos: o mapa diante do que não foi escrito para ele

Para medir a Parte 1 com a mesma régua da Parte 2, escrevemos `data/relatos_desafio.csv`:
**10 relatos novos**, um por doença do mapa mais dois que **não devem** gerar hipótese
(uma embolia pulmonar, que está fora do mapa, e uma consulta de rotina com negação). O
mapa foi **congelado** antes: nenhuma regra foi acrescentada depois de ver o resultado.

In [10]:
desafio = testar_desafio(mapa)
display(desafio[["esperado", "sugerido", "sintomas", "acertou", "o_que_testa"]])
diagnosticos = desafio[desafio.esperado != "(sem hipótese)"]
print(f"Acertos: {desafio.acertou.sum()}/{len(desafio)} no total · "
      f"{diagnosticos.acertou.sum()}/{len(diagnosticos)} nos relatos que pediam diagnóstico")

,esperado,sugerido,sintomas,acertou,o_que_testa
0,Angina,(sem hipótese),peso no peito,False,"angina descrita com outras palavras ('peso', 'some parado')"
1,Infarto agudo do miocárdio,(sem hipótese),,False,infarto sem as expressões 'dor no peito' e 'suor frio'
2,Insuficiência cardíaca,(sem hipótese),,False,"insuficiência cardíaca com sinônimos ('sem fôlego', 'tornozelos incham')"
3,Arritmia cardíaca,(sem hipótese),batedeira,False,arritmia em linguagem coloquial
4,AVC,(sem hipótese),,False,"AVC relatado por terceiro, com 'sem' que não é negação de sintoma"
5,Hipertensão arterial,(sem hipótese),,False,hipertensão sem a expressão 'pressão alta'
6,Pericardite,(sem hipótese),febre,False,pericardite com o quadro clássico em outras palavras
7,Estenose aórtica,(sem hipótese),,False,estenose com 'quase desmaiei' e 'sopro' soltos
8,(sem hipótese),(sem hipótese),falta de ar,True,embolia pulmonar: doença fora do mapa deve sair sem hipótese
9,(sem hipótese),(sem hipótese),,True,negação: nenhum sintoma presente


Acertos: 2/10 no total · 0/8 nos relatos que pediam diagnóstico


**De 10/10 para 0/8.** Nenhum dos oito relatos com doença recebeu diagnóstico. Os dois
acertos são os casos que deviam sair sem hipótese. O da consulta de rotina acertou pela
negação. O da embolia acertou pela regra de evidência mínima: o sistema reconheceu
"falta de ar", que sozinha não completa nenhuma regra. Sem essa regra, o desempate
teria devolvido "Arritmia cardíaca" para uma embolia pulmonar.

O padrão dos erros é um só: **o mapa reconhece frases, não sintomas.** "Peso no peito" e
"some depois de uns minutos parado" é angina para qualquer clínico, mas a regra de
angina pede "peso no peito" **junto com** "caminhada", e o paciente escreveu "caminho".
"Sem fôlego", "tornozelos incham", "rosto caído", "17 por 11": todos sinais claros, nenhum
no mapa. O sistema fez o que devia fazer quando não sabe (não chutou), mas **não sabe
quase nada fora das frases que nós mesmos escrevemos**.

É o mesmo resultado da Parte 2 por outro caminho: a métrica medida na própria base
media a nossa escrita, não o problema.

> **Um bug encontrado pelo desafio.** O relato "não sinto dor no peito **nem** falta de
> ar" extraía "falta de ar": a palavra *nem* não estava na lista de negações. Corrigimos
> a lista (`NEGACOES` em `src/extracao.py`). É uma correção do tratamento de negação,
> não do mapa: nenhuma regra foi alterada.

---
## 9. Onde este extrator quebra

Quatro limitações reais, encontradas rodando o próprio código — não hipóteses:

**1. Casamento por substring ignora morfologia.** O mapa tem `pernas inchadas`, o
relato diz "minhas pernas **estão inchadas**". São o mesmo sintoma para qualquer pessoa
e coisas diferentes para o `str.find` — o mesmo acontece com `barriga inchada` vs. "a
barriga está inchada". O remendo é cadastrar a variante à mão (foi o que fizemos com
`desmaio`/`desmaiar`), e é um remendo que não escala: cada verbo conjugado vira uma
linha nova no mapa.

**2. A negação é tratada por janela fixa de 3 palavras.** "Não sinto dor no peito"
funciona; "não sinto, desde que comecei o remédio, nenhuma dor no peito" não.

**3. A negação também erra para o lado oposto.** "*Nunca* senti uma dor no peito tão
forte" **afirma** a dor, mas o "nunca" logo antes faz o sistema descartá-la. Para uma
regra de janela, negação e intensificador são a mesma coisa.

**4. O mapa é o limite do sistema.** Ele só enxerga as 8 doenças e as 93 expressões
que nós escrevemos. Um relato de embolia pulmonar ou de dissecção de aorta cai fora do
mapa e sai sem hipótese (seção 8) — o que é preferível a um palpite errado, mas precisa ficar
explícito para quem for usar.

In [11]:
# A limitação nº 1, demonstrada:
for frase in ["minhas pernas estao inchadas", "pernas inchadas",
              "a barriga esta inchada", "barriga inchada",
              "cheguei a desmaiar", "desmaio"]:
    achado = sorted(encontrar_expressoes(frase, expressoes))
    print(f"{frase:<32} -> {achado if achado else '(nada encontrado)'}")

# A limitação nº 3: "nunca" como intensificador, lido como negação.
frase = "nunca senti uma dor no peito tão forte"
print(f"\n{frase} -> {sorted(encontrar_expressoes(frase, expressoes)) or '(nada encontrado)'}")

minhas pernas estao inchadas     -> (nada encontrado)
pernas inchadas                  -> ['pernas inchadas']
a barriga esta inchada           -> (nada encontrado)
barriga inchada                  -> ['barriga inchada']
cheguei a desmaiar               -> ['desmaiar']
desmaio                          -> ['desmaio']

nunca senti uma dor no peito tão forte -> (nada encontrado)


---
## 10. O que esta parte entrega

- `data/relatos_pacientes.txt` — 10 relatos completos
- `data/mapa_conhecimento.csv` — 79 regras, 8 doenças, no formato `sintoma_1 | sintoma_2 | doenca_associada`
- `src/extracao.py` — extração + ranqueamento, com negação e evidência explícita
- `data/relatos_desafio.csv` — 10 relatos novos, escritos com o mapa congelado
- **10/10** no gabarito interno e **0/8** nos diagnósticos dos relatos novos (2/2 nos que
  deviam sair sem hipótese). A distância entre os dois é o resultado desta parte.

Na Parte 2 o problema muda: em vez de saber *qual* doença, o sistema precisa decidir
*com que urgência* atender — e lá a regra escrita à mão dá lugar a um modelo treinado.